In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Thu May 11 15:28:01 2023

@author: James Lee
"""

import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import os
import copy
import math
import io
import numpy as np
from datetime import datetime
import copy

In [14]:
infile = open('universal_params', 'rb')
universal_params = pickle.load(infile)
infile.close

<function BufferedReader.close()>

In [17]:
data = {}
data["no_shocks_removed"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_no_shocks_removed")
data["all_shocks_removed"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_grpe_grpf_vu_shor")

data["grpe"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_grpe")
data["grpf"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_grpf")
data["vu"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_vu")
data["shortage"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_shortage")
data["lockdown"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name = "wo_residuals_lockdown")

# residuals + initial conditions
data["all_shocks_w_resid"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name="w_residuals_grpe_grpf_vu_short")
# initial conditions
data["pure_initial"] = pd.read_excel("all_data_decompositions.xlsx", sheet_name="wo_residuals_grpe_grpf_vu_shor")

data["residuals"] = pd.DataFrame()
data["residuals"]["period"] = data["pure_initial"]["period"]
data["residuals"]["contr_gw"] = data["all_shocks_w_resid"]["gw_simul"] - data["pure_initial"]["gw_simul"]
data["residuals"]["contr_gcpi"] = data["all_shocks_w_resid"]["gcpi_simul"] - data["pure_initial"]["gcpi_simul"]
data["residuals"]["contr_cf1"] = data["all_shocks_w_resid"]["cf1_simul"] - data["pure_initial"]["cf1_simul"]
data["residuals"]["contr_cf10"] = data["all_shocks_w_resid"]["cf10_simul"] - data["pure_initial"]["cf10_simul"]

# 1. Nejdříve vynutíme převod na DATETIME u všech tabulek
for shock in data.keys():
    # errors='coerce' zajistí, že i problematické texty se zkusí převést, nebo dají NaT
    data[shock]["period"] = pd.to_datetime(data[shock]["period"], errors='coerce')

# 2. Filtrování a Transformace (rolling)
# Musíme to udělat v tomto pořadí, aby všechny tabulky měly stejnou délku pro graf
for shock in data.keys():
    # Filtrujeme data (teď už je period typu datetime, takže to projde)
    data[shock] = data[shock][data[shock]["period"] >= datetime(2019, 10, 1)].copy()
    
    # Resetujeme index, aby rolling fungoval správně od nuly
    data[shock] = data[shock].reset_index(drop=True)
    
    # Schováme si period sloupec
    period_col = data[shock]["period"]
    
    # Vybereme jen číselné sloupce pro výpočet (zahodíme 'period')
    numeric_df = data[shock].select_dtypes(include=[np.number])

    if any(name in shock for name in ["gw", "gcpi"]) or shock in ["grpe", "grpf", "vu", "shortage", "lockdown", "residuals", "pure_initial", "no_shocks_removed", "all_shocks_removed"]:
        # Anualizovaný průměr (vyhladí zuby)
        transformed_df = numeric_df.rolling(window=4).mean()
    else:
        # Očekávání (cf1, cf10) - neprůměrujeme, ale posuneme o 3, aby seděla délka
        transformed_df = numeric_df.copy()
        transformed_df.iloc[:3, :] = np.nan 

    # Sestavíme tabulku zpět
    data[shock] = transformed_df
    data[shock]["period"] = period_col
    
    # Smažeme prvních 3 NaN řádky (u VŠECH tabulek stejně)
    data[shock] = data[shock].dropna().reset_index(drop=True)

# 3. Formátování x_label (děláme až teď, když máme finální řádky)
for shock in data.keys():
    data[shock]["x_label"] = ["Q" + str(itm.quarter) + " " + str(itm.year) for itm in data[shock]["period"]]
# Graph
plt.rcParams['font.family'] = 'sans-serif'
# plt.rcParams['font.sans-serif'] = ['Calibri']


font_size = 14
plt.rcParams['font.size'] = font_size
font = font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size' : 14}
plt.rc('font', **font)
plt.rc('figure', titlesize = font_size)
plt.rc('axes', labelsize = font_size)
plt.rc('xtick', labelsize = font_size)
plt.rc('xtick', labelsize = font_size)
plt.rc('legend', fontsize = 12)
plt.rc('axes', titlesize = font_size)

to_graph = ["gw", "gcpi", "cf1", "cf10"]
figure_dict = {"gw": "figure_13", "gcpi": "figure_12", "cf1": "figure_14", "cf10": "figure_15"}

'''
baseline denotes the start of where we want to start the bars from. If bars
are negative, we start from the bottom. If bars are positive, we start from
the top.

Top denotes position of top of stacked bars; and bottom denotes the bottom
value of the stacked bars
'''


for itm in to_graph:
    fig, ax = plt.subplots(figsize = universal_params['standard_size_PPT'])
    top = [0 for itm in data["all_shocks_removed"]["period"]]
    bottom = [0 for itm in data["all_shocks_removed"]["period"]]
    baseline = [0 for itm in data["all_shocks_removed"]["period"]]
    
    # Plot initial conditions
    series = data["pure_initial"][itm + "_simul"]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
    ax.bar(data["pure_initial"]["x_label"], series, 
        width = 0.4, color = [127/255.0, 127/255.0, 127/255.0], 
        bottom = baseline,label = "Initial conditions")
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]

    # Plot contribution of residuals   
    series = data["residuals"]["contr_" + itm]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
            
    ax.bar(data["residuals"]["x_label"], series, 
       width = 0.4, color = [80/255.0, 80/255.0, 80/255.0], # Tmavě šedá
       bottom = baseline, label = "Residuals")
       
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]
            
    # Plot contribution of vu
    series = data["vu"]["contr_" + itm]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
    ax.bar(data["vu"]["x_label"], series, 
       width = 0.4, color = [192/255.0, 0/255.0, 0/255.0], 
       bottom = baseline,label = "v/u")
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]

    # Plot contribution of grpe
    series = data["grpe"]["contr_" + itm]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
    ax.bar(data["grpe"]["x_label"], series, 
        width = 0.4, color = [46/255.0, 117/255.0, 182/255.0], 
        bottom = baseline,label = "Energy Prices")
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]
            
    # Plot contribution of grpf
    series = data["grpf"]["contr_" + itm]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
    ax.bar(data["grpf"]["x_label"], series, 
       width = 0.4, color = [157/255.0, 195/255.0, 230/255.0], 
       bottom = baseline,label = "Food Prices")
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]

    # Plot contribution of shortages
    series = data["shortage"]["contr_" + itm]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
    ax.bar(data["shortage"]["x_label"], series, 
       width = 0.4, color = [255/255.0, 217/255.0, 102/255.0], 
       bottom = baseline,label = "Shortages")
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]

    # --- LOCKDOWN ---
    series = data["lockdown"]["contr_" + itm]
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            baseline[idx] = top[idx]
        else:
            baseline[idx] = bottom[idx]
    
    ax.bar(data["lockdown"]["x_label"], series, 
       width = 0.4, color = [112/255.0, 48/255.0, 160/255.0], # Fialová barva
       bottom = baseline, label = "Lockdowns")
    
    temp = copy.deepcopy(top)
    for idx in range(len(series)):
        if series.iloc[idx] >= 0:
            top[idx] += series.iloc[idx]
            bottom[idx] += 0
        else:
            top[idx] += 0
            bottom[idx] += series.iloc[idx]
    # ---------------------------------------
    # Plot Actual Data Series
    if itm == "gw":
        ax.plot(data["no_shocks_removed"]["x_label"], data["no_shocks_removed"][itm], 
            lw = 3, color = [0/255.0, 0/255.0, 0/255.0], label = "Actual Wage Growth")
    elif itm == "gcpi":
        ax.plot(data["no_shocks_removed"]["x_label"], data["no_shocks_removed"][itm], 
            lw = 3, color = [0/255.0, 0/255.0, 0/255.0], label = "Actual Price Inflation")
    elif itm == "cf1":
        ax.plot(data["no_shocks_removed"]["x_label"], data["no_shocks_removed"][itm], 
            lw = 3, color = [0/255.0, 0/255.0, 0/255.0], label = "Actual 1-Year Expectations")
    elif itm == "cf10":
        ax.plot(data["no_shocks_removed"]["x_label"], data["no_shocks_removed"][itm], 
            lw = 3, color = [0/255.0, 0/255.0, 0/255.0], label = "Actual 10-Year Expectations")
            
# Show ONLY Q1 labels (Schová vše kromě prvních kvartálů)
    xlabels = ax.get_xticklabels()
    for label in xlabels:
        # Získáme text popisku (např. "Q1 2020")
        text = label.get_text()
        
        # Pokud text NEZAČÍNÁ na "Q1", schováme ho
        if not text.startswith("Q1"):
            label.set_visible(False)
        else:
            # Smaže "Q1 " a nechá jen čistý rok
            label.set_text(text.replace("Q1 ", ""))
            
    # Pro aplikování upraveného textu (bez Q1) do grafu:
    ax.set_xticklabels([label.get_text() for label in xlabels])
    
    # Other formatting
    current_bottom, current_top = ax.get_ylim()
    
    if itm == "cf1":
        ax.set_ylim(current_bottom - 5, math.ceil(current_top))
    elif itm == "cf10":
        ax.set_ylim(current_bottom - 1.5, math.ceil(current_top))
    elif itm == "gw":
        ax.set_ylim(current_bottom - 6, math.ceil(current_top))
    elif itm == "gcpi":
        ax.set_ylim(current_bottom - 7, math.ceil(current_top))
    else:
        ax.set_ylim(current_bottom, math.ceil(current_top))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.grid(True, which = 'minor')
    ax.set_axisbelow(True) # puts grid lines in the back
    ax.set_ylabel("Percent")
    
    legend = ax.legend(frameon = True, labelspacing = 0.1, 
                       bbox_to_anchor=(0.99, 0.01), loc = 'lower right', ncol=2, framealpha = 1.0)
    
    frame = legend.get_frame()
    frame.set_edgecolor("black")
    frame.set_facecolor('white')
    plt.grid(visible = True, axis = 'y', linestyle = 'dotted', linewidth = 1.5)

    #Save PDF
    pdf_filename = f"{figure_dict[itm]}_{itm}_decomp.pdf"
    fig.savefig(pdf_filename, format='pdf', bbox_inches='tight')
    plt.close(fig)
    
    print(f"Uloženo: {pdf_filename}")

/var/folders/tz/y0sqk00n3_bd7tlckl9zbspw0000gn/T/ipykernel_98226/795839119.py:263: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([label.get_text() for label in xlabels])


Uloženo: figure_13_gw_decomp.pdf


/var/folders/tz/y0sqk00n3_bd7tlckl9zbspw0000gn/T/ipykernel_98226/795839119.py:263: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([label.get_text() for label in xlabels])


Uloženo: figure_12_gcpi_decomp.pdf


/var/folders/tz/y0sqk00n3_bd7tlckl9zbspw0000gn/T/ipykernel_98226/795839119.py:263: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([label.get_text() for label in xlabels])


Uloženo: figure_14_cf1_decomp.pdf


/var/folders/tz/y0sqk00n3_bd7tlckl9zbspw0000gn/T/ipykernel_98226/795839119.py:263: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([label.get_text() for label in xlabels])


Uloženo: figure_15_cf10_decomp.pdf
